# 01 - Data Extraction

**Energy Transition & Electric Mobility in Portugal**

First step of the pipeline. Downloads four public sources into `data/raw/`:

| Source | Endpoint | Granularity | Output file |
|---|---|---|---|
| REN Data Hub | `ElectricityProductionBreakdownDaily` | 15 minutes, one series per source | `ren_production_breakdown_raw.json` |
| REN Data Hub | `ElectricityMarketPricesDaily` | hourly until June 2026, then every 15 minutes; Portugal and Spain | `ren_market_prices_raw.json` |
| E-REDES Open Data | `consumo_horario_mobilidade_eletrica` (2 datasets) | monthly per municipality, summed on the server | `eredes_mobility_grouped_raw.json` |
| INE | resident population estimates (annual) | per municipality, latest year available | `ine_population_raw.json` |

The extraction is incremental. Every run moves the window forward to yesterday and only requests the days that are not saved yet, so re-running the four notebooks in order is all it takes to update the project. A first run on an empty `data/raw/` folder takes about 20 to 25 minutes per REN endpoint; later runs take a minute or two.

## Setup

In [1]:
import json
import time
import unicodedata
from datetime import date, timedelta
from pathlib import Path

import requests
from dateutil.relativedelta import relativedelta

# Works whether the notebook sits in notebooks/ or in the project root
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_RAW.mkdir(parents=True, exist_ok=True)

REN_BASE_URL = "https://servicebus.ren.pt/datahubapi/electricity"
EREDES_BASE_URL = "https://e-redes.opendatasoft.com/api/explore/v2.1/catalog/datasets"

print(f"Run date: {date.today()}")
print(f"Raw data folder: {DATA_RAW.relative_to(PROJECT_ROOT)}")

Run date: 2026-10-29
Raw data folder: data\raw


## Extraction window

The window always ends yesterday (the current day is often not fully published) and covers the last `YEARS_OF_HISTORY` years. Both REN endpoints use the same `date_range`, so production and prices always refer to the same days. Days that fall out of the window on a later run are removed from the raw files.

E-REDES does not use this window: the portal only keeps about the last 12 months (see below).

In [2]:
YEARS_OF_HISTORY = 5

END_DATE = date.today() - timedelta(days=1)
START_DATE = END_DATE - relativedelta(years=YEARS_OF_HISTORY)
date_range = [
    START_DATE + timedelta(days=i) for i in range((END_DATE - START_DATE).days + 1)
]

print(f"REN window: {START_DATE} to {END_DATE} ({len(date_range)} days)")

REN window: 2021-10-28 to 2026-10-28 (1827 days)


## REN extraction helpers

`extract_ren_endpoint` works in four steps:

1. Load the days already saved, if the file exists.
2. Request only the days of the window that are missing.
3. Retry gaps that sit *before* the latest saved day. Single-day errors from this API have always been temporary, so a few retries usually recover them. Days *after* the latest saved day are treated as "not published yet".
4. Save the days inside the window, sorted by date. This step also runs if the kernel is interrupted, so downloaded days are never lost.

In [3]:
def fetch_ren_day(endpoint, day, delay=0.5):
    """Request one day from a REN endpoint. Returns the JSON payload, or None if the request fails."""
    try:
        response = requests.get(
            f"{REN_BASE_URL}/{endpoint}",
            params={"culture": "en-US", "date": day},
            timeout=30,
        )
        payload = response.json() if response.status_code == 200 else None
    except (requests.RequestException, ValueError):
        payload = None
    time.sleep(delay)  # the API has no documented rate limit, so keep a small pause
    return payload


def save_raw(path, saved, wanted_days):
    """Write the saved days that fall inside the window to disk, sorted by date."""
    keep = set(wanted_days)
    entries = [saved[d] for d in sorted(saved) if d in keep]
    with open(path, "w") as f:
        json.dump(entries, f, separators=(",", ":"))
    return entries


def extract_ren_endpoint(endpoint, filename, days, retries=3):
    """Incremental extraction of one REN daily endpoint (steps described above)."""
    path = DATA_RAW / filename
    wanted = [d.isoformat() for d in days]

    saved = {}
    if path.exists():
        with open(path) as f:
            saved = {entry["date"]: entry for entry in json.load(f)}
    to_request = [d for d in wanted if d not in saved]
    print(f"{filename}: {len(saved)} days already saved, {len(to_request)} to request")

    latest = None
    try:
        for day in to_request:
            payload = fetch_ren_day(endpoint, day)
            if payload is not None:
                saved[day] = {"date": day, "data": payload}

        latest = max(saved) if saved else None
        interior_gaps = [d for d in wanted if d not in saved and latest and d < latest]
        for day in interior_gaps:
            for attempt in range(1, retries + 1):
                payload = fetch_ren_day(endpoint, day, delay=1.0)
                if payload is not None:
                    saved[day] = {"date": day, "data": payload}
                    print(f"  recovered {day} on retry {attempt}")
                    break
    finally:
        entries = save_raw(path, saved, wanted)

    still_missing = [d for d in wanted if d not in saved]
    gaps_left = [d for d in still_missing if latest and d < latest]
    print(f"  saved {len(entries)} days | missing {len(still_missing)}")
    if gaps_left:
        print(f"  gaps that could not be recovered: {gaps_left}")
    if still_missing:
        print(f"  not published yet: {still_missing[0]} to {still_missing[-1]}")
    return entries

## REN: production

One call per day returns every source as a separate series (Hydro, Solar, Wind, Natural Gas, Biomass, Coal, Wave, Import, Export, batteries, consumption and others), each with one reading in MW every 15 minutes.

In [4]:
ren_production = extract_ren_endpoint(
    "ElectricityProductionBreakdownDaily",
    "ren_production_breakdown_raw.json",
    date_range,
)

ren_production_breakdown_raw.json: 1827 days already saved, 1 to request
  saved 1827 days | missing 0


## REN: market prices

Two series per day (Portugal and Spain, in EUR/MWh). REN reports 24 hourly values per day up to 29 June 2026 and 96 values (one every 15 minutes) from 30 June 2026. Prices are sometimes published late; days that are not available yet are requested again on the next run.

In [5]:
ren_prices = extract_ren_endpoint(
    "ElectricityMarketPricesDaily",
    "ren_market_prices_raw.json",
    date_range,
)

ren_market_prices_raw.json: 1827 days already saved, 1 to request
  saved 1827 days | missing 0


## E-REDES: electric mobility consumption

Two datasets cover the 18 mainland districts between them (`_01_a_10` and `_11_a_18`). Energy is summed on the server by year, month, district and municipality, instead of downloading millions of hourly rows. Pagination stops when a page comes back empty, because `total_count` is not reliable on `group_by` queries.

E-REDES only keeps about the last 12 months online. To build a longer history over time, each run merges the download with the file already saved: months still online are refreshed, and months that have left the portal are kept.

In [6]:
def fetch_eredes_grouped(dataset_id, page_size=200):
    """Monthly energy per municipality, summed on the server and paginated until an empty page."""
    rows, offset = [], 0
    while True:
        response = requests.get(
            f"{EREDES_BASE_URL}/{dataset_id}/records",
            params={
                "select": "sum(energia_consumida) as total_energia_kwh",
                "group_by": "ano,mes,distrito,concelho",
                "limit": page_size,
                "offset": offset,
            },
            timeout=60,
        )
        response.raise_for_status()
        page = response.json()["results"]
        if not page:
            return rows
        rows.extend(page)
        offset += page_size
        time.sleep(0.3)


def eredes_key(row):
    return (row["ano"], row["mes"], row["distrito"], row["concelho"])


EREDES_DATASETS = [
    "consumo_horario_mobilidade_eletrica_01_a_10",
    "consumo_horario_mobilidade_eletrica_11_a_18",
]

downloaded = []
for dataset_id in EREDES_DATASETS:
    rows = fetch_eredes_grouped(dataset_id)
    print(f"{dataset_id}: {len(rows)} grouped rows")
    downloaded.extend(rows)

# The two datasets must not overlap, otherwise energy would be counted twice
new_keys = {eredes_key(r) for r in downloaded}
assert len(new_keys) == len(downloaded), "The two E-REDES datasets overlap"

eredes_path = DATA_RAW / "eredes_mobility_grouped_raw.json"
saved_rows = []
if eredes_path.exists():
    with open(eredes_path) as f:
        saved_rows = json.load(f)

merged = {eredes_key(r): r for r in saved_rows}
kept_from_earlier_runs = len(set(merged) - new_keys)
merged.update({eredes_key(r): r for r in downloaded})

eredes_rows = sorted(
    merged.values(),
    key=lambda r: (int(r["ano"]), int(r["mes"]), r["distrito"], r["concelho"]),
)
with open(eredes_path, "w") as f:
    json.dump(eredes_rows, f, separators=(",", ":"))

print(
    f"Saved {len(eredes_rows)} rows ({kept_from_earlier_runs} kept from earlier runs, no longer online)"
)

consumo_horario_mobilidade_eletrica_01_a_10: 1764 grouped rows
consumo_horario_mobilidade_eletrica_11_a_18: 1572 grouped rows
Saved 3336 rows (0 kept from earlier runs, no longer online)


## INE: resident population by municipality

Used by the dashboard to show electric mobility consumption per resident. The request asks for the totals by sex and age group; if that filter is not accepted, the full breakdown is downloaded instead and `02_cleaning.ipynb` keeps the largest value per place, which is always the total.

The most recent year that covers the mainland municipalities is used. If no request works, the rest of the pipeline still runs and the dashboard shows totals only.

In [ ]:
INE_API_URL = "https://www.ine.pt/ine/json_indicador/pindica.jsp"
# Annual estimates of the resident population by place of residence, sex and age group
INE_POPULATION_INDICATORS = ["0008273", "0008272"]
MIN_MUNICIPALITIES = 270  # of the 278 mainland municipalities


def name_key(name):
    """Upper case, no accents, single spaces: used to match place names across sources."""
    plain = unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode()
    return " ".join(plain.upper().replace("-", " ").split())


def fetch_ine_records(indicator, year, totals_only):
    """One request to the INE API. Returns the records for that year, or None."""
    params = {"op": "2", "varcd": indicator, "Dim1": f"S7A{year}", "lang": "PT"}
    if totals_only:
        params.update({"Dim3": "T", "Dim4": "T"})  # sex and age group totals
    try:
        response = requests.get(INE_API_URL, params=params, timeout=90)
        records = response.json()[0]["Dados"][str(year)]
    except (requests.RequestException, ValueError, KeyError, IndexError, TypeError):
        return None
    finally:
        time.sleep(0.5)
    return records if isinstance(records, list) and records else None


def find_population(mainland_keys):
    """Try the most recent years first and stop at the first answer that covers the mainland."""
    for year in range(date.today().year - 1, date.today().year - 6, -1):
        for indicator in INE_POPULATION_INDICATORS:
            for totals_only in (True, False):
                records = fetch_ine_records(indicator, year, totals_only)
                if not records:
                    continue
                matched = {
                    name_key(r.get("geodsg", "")) for r in records
                } & mainland_keys
                print(
                    f"INE {indicator} {year} (totals filter: {totals_only}): "
                    f"{len(records)} records, {len(matched)} mainland municipalities"
                )
                if len(matched) >= MIN_MUNICIPALITIES:
                    return {"indicator": indicator, "year": year, "records": records}
    return None


ine_population = find_population({name_key(r["concelho"]) for r in eredes_rows})
if ine_population:
    with open(DATA_RAW / "ine_population_raw.json", "w") as f:
        json.dump(ine_population, f, separators=(",", ":"))
    print(
        f"Saved INE population, indicator {ine_population['indicator']}, year {ine_population['year']}"
    )
else:
    print(
        "INE population not available. The dashboard will show totals only; "
        "the previous ine_population_raw.json, if any, is kept."
    )

INE 0008273 2023 (totals filter: True): 344 records, 278 mainland municipalities
Saved INE population, indicator 0008273, year 2023


## Validation summary

In [8]:
def describe_daily(name, entries):
    days = sorted(date.fromisoformat(e["date"]) for e in entries)
    expected = (days[-1] - days[0]).days + 1
    print(
        f"{name}: {len(days)} days, {days[0]} to {days[-1]}, gaps inside the range: {expected - len(days)}"
    )


describe_daily("REN production", ren_production)
describe_daily("REN prices    ", ren_prices)

months = sorted({(int(r["ano"]), int(r["mes"])) for r in eredes_rows})
districts = {r["distrito"] for r in eredes_rows}
municipalities = {r["concelho"] for r in eredes_rows}
print(
    f"E-REDES       : {len(eredes_rows)} rows, {months[0][0]}-{months[0][1]:02d} to {months[-1][0]}-{months[-1][1]:02d}, "
    f"{len(districts)} districts, {len(municipalities)} municipalities"
)

REN production: 1827 days, 2021-10-28 to 2026-10-28, gaps inside the range: 0
REN prices    : 1827 days, 2021-10-28 to 2026-10-28, gaps inside the range: 0
E-REDES       : 3336 rows, 2025-05 to 2026-04, 18 districts, 278 municipalities


## Notes on the APIs

- **REN** returns chart-style JSON (`xAxis.categories` plus a list of `series`), not tables. Production has 96 readings per series on a normal day (92 or 100 on the days the clocks change). Prices had 24 hourly values per day until 29 June 2026 and have 96 since then.
- **REN** has real data back to at least 2010. Isolated errors on single days are temporary and recover on retry. Prices for recent months have at times returned errors for weeks before being published in bulk.
- **REN `Wave`** is reported as 0 on every day: wave energy is not in commercial use in Portugal.
- **E-REDES** is an OpenDataSoft portal. `total_count` is unreliable on `group_by` queries, and a `where` filter on the `data` date field does not filter (the `ano` and `mes` text fields do).
- **E-REDES** splits mobility consumption by groups of districts. The dataset without a suffix returns HTTP 400, so `_01_a_10` and `_11_a_18` are used. The Azores and Madeira are not covered.
- The raw production file is around 20 MB, so `data/raw/` is kept out of version control.
- **INE** is queried through its JSON API (`pindica.jsp`). Names are matched to the E-REDES municipalities without accents or hyphens; mainland codes start with `1`, which separates duplicates such as Lagoa (Algarve and Azores).